# 1. Setting Up GABRIEL and Loading Data

In [ ]:
#Load the GABRIEL library

%pip install openai-gabriel

In [ ]:
#Import GABRIEL and set up OpenAI API key

import gabriel
from gabriel.utils.plot_utils import regression_plot, bar_plot, box_plot, line_plot
import os
import pandas as pd

os.environ['OPENAI_API_KEY'] = 

In [ ]:
#Import files into the runtime, then load them here
##This loaded the dataframe of all of my articles before duplicates were removed

df_articles = pd.read_csv("/content/articles_by_date.csv")

In [ ]:
#The complete dataframe of all articles after duplicates were removed
df_articles_3 = pd.read_csv("/content/FINAL-articles-by-date-deduped.csv")

In [ ]:
#This is the DF for the sample of 151 articles that have the search term in the title, not just in the article body
df_articles_2 = pd.read_csv("articles_gfc_in_title.csv")

In [ ]:
#This is the cleaned dataframe for the classifications for articles blaming the US
import pandas as pd

df_blame = pd.read_csv("/content/Blame-Classified-no-nulls-Complete.csv")

In [ ]:
#This is the dataframe I created manually with my own classifications for articles blaming the US
import pandas as pd

df_manual = pd.read_csv("/content/headlines-blame-results-manual.csv")

In [ ]:
#This is the dataframe for the first set of ratings
import pandas as pd

df_failure = pd.read_csv("/content/rate_1_cleaned.csv")

In [ ]:
#This is the dataframe for more classifications
import pandas as pd

df_decline_opp = pd.read_csv("/content/Decline_Opp_Classified-no-nulls.csv")

In [ ]:
#The dataframes for more ratings
df_rate8 = pd.read_csv("/content/Rate-8-atts.csv")

In [ ]:
import pandas as pd

df_rate9 = pd.read_csv("/content/rate8-pt2.csv")

In [ ]:
#Dataframe for my last set of classifications
import pandas as pd

classify_results = pd.read_csv("/content/classify-4-14.csv")

# 2. Cleaning the Data

In [ ]:
#Original dataframe containing 10,192 articles
df_articles

In [ ]:
#Check to see how many rows there are, how many unique article texts there are, and how many duplicate texts there are
print("Total rows:", len(df_articles))
print("Unique texts:", df_articles["text"].nunique())
print("Duplicate rows:", len(df_articles) - df_articles["text"].nunique())

In [ ]:
#Drop duplicates and check again
df_articles_clean = df_articles.drop_duplicates(subset=["text"]).copy()

print("Original:", len(df_articles))
print("Clean:", len(df_articles_clean))
print("Removed:", len(df_articles) - len(df_articles_clean))

In [ ]:
#Check to see if there are any article text columns that contain "nulls" (an article without any text and only an image)
df_articles_clean[df_articles_clean["text"].isna()]

In [ ]:
#Remove articles that contain no text and confirm there are no "Null" values in the article text column
df_articles_clean = df_articles_clean.dropna(subset=["text"]).drop_duplicates(subset=["text"]).copy()

df_articles_clean["text"].isna().sum()

In [ ]:
#Correct the "id" and "index" so that rows and articles are numbered correctly now that articles have been removed from the df
df_articles_clean = df_articles_clean.reset_index(drop=True)
df_articles_clean["id"] = range(1, len(df_articles_clean) + 1)

#Confirm that this was successful by looking at the bottom five rows of the df
df_articles_clean.tail(5)

In [ ]:
#Turn this cleaned df into a CSV file and download the new cleaned dataframe containing 10,148 articles
df_articles_clean.to_csv("FINAL-articles_by_date_deduplicated.csv", index=False)

from google.colab import files
files.download("FINAL-articles_by_date_deduplicated.csv")

In [ ]:
#In the future this new cleaned df with all 10,148 articles is referred to as df_articles_3
df_articles_3

# 3. Manual Classifications vs. GABRIEL Classifications

In [ ]:
#Run one attribute on a sample of 151 articles
labels = {
    "blames US": "The text attributes blame for the Global Financial Crisis to the US or describes the Global Financial Crisis as the result of the US subprime mortgage crisis, Lehman Brothers declaring bankruptcy, or other references to a crisis in the US that spread."
}

results = await gabriel.classify(
    df_articles_2,
    column_name = "text",       # name of column with text to classify
    labels = labels,            # dictionary of label definitions, defined above
    save_dir = "headlines_classify_blame_9",  # directory to save results, use a Google Drive folder (e.g. "/content/drive/folder") for permanent storage (see 'Loading your data' section)
    model = "gpt-5.4-mini",     # GPT model used for classification (gpt-5.4 is best, gpt-5.4-nano is cheapest and fastest, gpt-5.4-mini is good balance)
    modality = "text",  # input modality can also be "entity" (for terms like 'apple pie', not full texts), "web" (see web section), "pdf", "image", or "audio" (see multimodal section)
    ###
    ### the parameters below are less important
    ###
    n_runs = 1,                 # number of classification passes per text
    n_attributes_per_run = 8,   # if more than 8 labels, will split into separate calls of <= 8
    n_parallels = 20,          # max parallel threads (reduce if many errors, increase for higher speed)
    reset_files = False, # rerunning the cell loads from save / picks up from checkpoint
)

In [ ]:
#Confirm no nulls in the sample
results["blames US"].isna().sum()

In [ ]:
#Confirm no duplicates in the sample
df_articles_2["text"].duplicated().any()

In [ ]:
#Download classifications for sample
results.to_csv("results.csv", index=False)

from google.colab import files
files.download("results.csv")

In [ ]:
#On my local computer I input my manual classifications to the dataframe and reimported it
#df_manual is a df with both manual classifications and AI's classifications
df_manual

In [ ]:
#See percentage of rows where articles have the same classification values
 ((df_manual["blames US"] == df_manual["Manual"]).mean()) * 100

In [ ]:
#Visualize the difference with a pie chart
labels = ["Match", "Mismatch"]
sizes = [

    (df_manual["blames US"] == df_manual["Manual"]).sum(),
    (df_manual["blames US"] != df_manual["Manual"]).sum()
]

plt.pie(sizes, labels=labels, autopct="%1.1f%%")
plt.title("Classification Agreement: Manual vs. GABRIEL")
plt.show()

# 4. Testing for Rate Reliability

In [ ]:
#Test using sample of articles (run #1)

import pandas as pd

# Define the features we want to measure
attributes = {
    "Tone": "The overall sentiment or tone of the article (0 = extremely negative or pessimistic, 50 = neutral/mixed, 100 = extremely positive or optimistic)."
}

additional_instructions = "" #### Any additional instructions you want to pass to the model when rating. For instance examples of how to rate the text

# Make the call to the package
results_1 = await gabriel.rate(
    df = df_articles_2,     # sample of 151 articles
    column_name = "text",       # name of column with text to rate
    attributes = attributes,    # attributes to score on 0–100 scale; defined above
    save_dir = "Test-Rate-5",      # directory to save results, use a Google Drive folder (e.g. "/content/drive/folder") for permanent storage (see 'Loading your data' section)
    model = "gpt-5.4-mini",     # GPT model used for ratings (gpt-5.4 is best, gpt-5.4-nano is cheapest and fastest, gpt-5.4-mini is good balance)
    modality = "text",  # input modality can also be "entity" (for terms like 'apple pie', not full texts), "web" (see web section), "pdf", "image", or "audio" (see multimodal section)
    ###
    ### the parameters below are less important
    ###
    additional_instructions = additional_instructions,
    n_runs = 1,                 # number of rating passes per text (higher = averaged over more ratings)
    n_attributes_per_run = 8,   # if more than 8 attributes, will split into separate calls of <= 8
    n_parallels = 50,          # max parallel threads (reduce if many errors, increase for higher speed)
    reasoning_effort = 'low',   # amount of reasoning effort, 'none' is default
    reset_files = False  # if False, rerunning the cell loads from save / picks up from checkpoint
)

In [ ]:
# Run #2
attributes = {
    "Crisis Attitude": "The text describes the impact of the Global Financial Crisis on China, including the Chinese government, economy, firms, and society (0 = extremely negative or pessimistic, 50 = neutral/mixed, 100 = extremely positive or optimistic).",
    "Tone": "The overall sentiment or tone of the article (0 = extremely negative or pessimistic, 50 = neutral/mixed, 100 = extremely positive or optimistic)."
}

additional_instructions = "" #### Any additional instructions you want to pass to the model when rating. For instance examples of how to rate the text

# Make the call to the package
results_2 = await gabriel.rate(
    df = df_articles_2,      # sample of 151 articles
    column_name = "text",       # name of column with text to rate
    attributes = attributes,    # attributes to score on 0–100 scale; defined above
    save_dir = "Test-Rate-6",      # directory to save results, use a Google Drive folder (e.g. "/content/drive/folder") for permanent storage (see 'Loading your data' section)
    model = "gpt-5.4-mini",     # GPT model used for ratings (gpt-5.4 is best, gpt-5.4-nano is cheapest and fastest, gpt-5.4-mini is good balance)
    modality = "text",  # input modality can also be "entity" (for terms like 'apple pie', not full texts), "web" (see web section), "pdf", "image", or "audio" (see multimodal section)
    ###
    ### the parameters below are less important
    ###
    additional_instructions = additional_instructions,
    n_runs = 1,                 # number of rating passes per text (higher = averaged over more ratings)
    n_attributes_per_run = 8,   # if more than 8 attributes, will split into separate calls of <= 8
    n_parallels = 50,          # max parallel threads (reduce if many errors, increase for higher speed)
    reasoning_effort = 'low',   # amount of reasoning effort, 'none' is default
    reset_files = False  # if False, rerunning the cell loads from save / picks up from checkpoint
)

In [ ]:
# Run #3
attributes = {
    "Crisis Attitude": "The text describes the impact of the Global Financial Crisis on China, including the Chinese government, economy, firms, and society (0 = extremely negative or pessimistic, 50 = neutral/mixed, 100 = extremely positive or optimistic).",
    "Tone": "The overall sentiment or tone of the article (0 = extremely negative or pessimistic, 50 = neutral/mixed, 100 = extremely positive or optimistic)."
}

additional_instructions = "" #### Any additional instructions you want to pass to the model when rating. For instance examples of how to rate the text

# Make the call to the package
results_3 = await gabriel.rate(
    df = df_articles_2,      # sample of 151 articles
    column_name = "text",       # name of column with text to rate
    attributes = attributes,    # attributes to score on 0–100 scale; defined above
    save_dir = "Test-Rate-7",      # directory to save results, use a Google Drive folder (e.g. "/content/drive/folder") for permanent storage (see 'Loading your data' section)
    model = "gpt-5.4-mini",     # GPT model used for ratings (gpt-5.4 is best, gpt-5.4-nano is cheapest and fastest, gpt-5.4-mini is good balance)
    modality = "text",  # input modality can also be "entity" (for terms like 'apple pie', not full texts), "web" (see web section), "pdf", "image", or "audio" (see multimodal section)
    ###
    ### the parameters below are less important
    ###
    additional_instructions = additional_instructions,
    n_runs = 1,                 # number of rating passes per text (higher = averaged over more ratings)
    n_attributes_per_run = 8,   # if more than 8 attributes, will split into separate calls of <= 8
    n_parallels = 50,          # max parallel threads (reduce if many errors, increase for higher speed)
    reasoning_effort = 'low',   # amount of reasoning effort, 'none' is default
    reset_files = False  # if False, rerunning the cell loads from save / picks up from checkpoint
)

In [ ]:
# Run #4
attributes = {
    "Crisis Attitude": "The text describes the impact of the Global Financial Crisis on China, including the Chinese government, economy, firms, and society (0 = extremely negative or pessimistic, 50 = neutral/mixed, 100 = extremely positive or optimistic).",
    "Tone": "The overall sentiment or tone of the article (0 = extremely negative or pessimistic, 50 = neutral/mixed, 100 = extremely positive or optimistic)."
}

additional_instructions = "" #### Any additional instructions you want to pass to the model when rating. For instance examples of how to rate the text

# Make the call to the package
results_4 = await gabriel.rate(
    df = df_articles_2,      # sample of 151 articles
    column_name = "text",       # name of column with text to rate
    attributes = attributes,    # attributes to score on 0–100 scale; defined above
    save_dir = "Test-Rate-8",      # directory to save results, use a Google Drive folder (e.g. "/content/drive/folder") for permanent storage (see 'Loading your data' section)
    model = "gpt-5.4-mini",     # GPT model used for ratings (gpt-5.4 is best, gpt-5.4-nano is cheapest and fastest, gpt-5.4-mini is good balance)
    modality = "text",  # input modality can also be "entity" (for terms like 'apple pie', not full texts), "web" (see web section), "pdf", "image", or "audio" (see multimodal section)
    ###
    ### the parameters below are less important
    ###
    additional_instructions = additional_instructions,
    n_runs = 1,                 # number of rating passes per text (higher = averaged over more ratings)
    n_attributes_per_run = 8,   # if more than 8 attributes, will split into separate calls of <= 8
    n_parallels = 50,          # max parallel threads (reduce if many errors, increase for higher speed)
    reasoning_effort = 'low',   # amount of reasoning effort, 'none' is default
    reset_files = False  # if False, rerunning the cell loads from save / picks up from checkpoint
)

In [ ]:
#Run #5
# Define the features we want to measure
attributes = {
    "Crisis Attitude": "The text describes the impact of the Global Financial Crisis on China, including the Chinese government, economy, firms, and society (0 = extremely negative or pessimistic, 50 = neutral/mixed, 100 = extremely positive or optimistic).",
    "Tone": "The overall sentiment or tone of the article (0 = extremely negative or pessimistic, 50 = neutral/mixed, 100 = extremely positive or optimistic)."
}

additional_instructions = "" #### Any additional instructions you want to pass to the model when rating. For instance examples of how to rate the text

# Make the call to the package
results_5 = await gabriel.rate(
    df = df_articles_2,      # sample of 151 articles
    column_name = "text",       # name of column with text to rate
    attributes = attributes,    # attributes to score on 0–100 scale; defined above
    save_dir = "Test-Rate-9",      # directory to save results, use a Google Drive folder (e.g. "/content/drive/folder") for permanent storage (see 'Loading your data' section)
    model = "gpt-5.4-mini",     # GPT model used for ratings (gpt-5.4 is best, gpt-5.4-nano is cheapest and fastest, gpt-5.4-mini is good balance)
    modality = "text",  # input modality can also be "entity" (for terms like 'apple pie', not full texts), "web" (see web section), "pdf", "image", or "audio" (see multimodal section)
    ###
    ### the parameters below are less important
    ###
    additional_instructions = additional_instructions,
    n_runs = 1,                 # number of rating passes per text (higher = averaged over more ratings)
    n_attributes_per_run = 8,   # if more than 8 attributes, will split into separate calls of <= 8
    n_parallels = 50,          # max parallel threads (reduce if many errors, increase for higher speed)
    reasoning_effort = 'low',   # amount of reasoning effort, 'none' is default
    reset_files = False  # if False, rerunning the cell loads from save / picks up from checkpoint
)

In [ ]:
# Remove "Crisis Attitude" from results_1 through results_5

for df_name in ["results_1", "results_2", "results_3", "results_4", "results_5"]:
    globals()[df_name].drop(columns=["Crisis Attitude"], inplace=True, errors="ignore")

In [ ]:
# Add Tone columns from results_2 through results_5 into results_1

results_1["Tone_2"] = results_2["Tone"]
results_1["Tone_3"] = results_3["Tone"]
results_1["Tone_4"] = results_4["Tone"]
results_1["Tone_5"] = results_5["Tone"]

In [ ]:
results_2.head(5)

In [ ]:
results_3.head(5)

In [ ]:
results_1

In [ ]:
#take average rating for each article across 5 draws for a given year to compute CI
import pandas as pd
import numpy as np
from scipy import stats

tone_cols = ["Tone", "Tone_2", "Tone_3", "Tone_4", "Tone_5"]

rows = []

for year, g in results_1.groupby("year"):
    draw_means = g[tone_cols].mean(axis=0).values   # 5 means, one per draw

    mean_est = draw_means.mean()
    sd = draw_means.std(ddof=1)
    se = sd / np.sqrt(5)

    t_crit = stats.t.ppf(0.975, df=4)

    ci_low = mean_est - t_crit * se
    ci_high = mean_est + t_crit * se

    rows.append({
        "year": year,
        "mean_tone": mean_est,
        "ci_low": ci_low,
        "ci_high": ci_high
    })

year_summary = pd.DataFrame(rows)
year_summary

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plt.plot(year_summary["year"], year_summary["mean_tone"], marker="o")

plt.fill_between(
    year_summary["year"],
    year_summary["ci_low"],
    year_summary["ci_high"],
    alpha=0.25
)

plt.xlabel("Year")
plt.ylabel("Average Tone")
plt.title("Average Tone by Year with 95% Confidence Intervals")
plt.show()

# 4. First Set of Classifications

##Blames US

In [ ]:
#Originally I ran this classification prompt on all 10,192 articles before the df was cleaned.
labels = {
    "blames US": "The text attributes blame for the Global Financial Crisis to the US or describes the Global Financial Crisis as the result of the US subprime mortgage crisis, Lehman Brothers declaring bankruptcy, or other references to a crisis in the US that spread."
}

results = await gabriel.classify(
    df_articles,
    column_name = "text",       # name of column with text to classify
    labels = labels,            # dictionary of label definitions, defined above
    save_dir = "headlines_classify_blame",  # directory to save results, use a Google Drive folder (e.g. "/content/drive/folder") for permanent storage (see 'Loading your data' section)
    model = "gpt-5.4-mini",     # GPT model used for classification (gpt-5.4 is best, gpt-5.4-nano is cheapest and fastest, gpt-5.4-mini is good balance)
    modality = "text",  # input modality can also be "entity" (for terms like 'apple pie', not full texts), "web" (see web section), "pdf", "image", or "audio" (see multimodal section)
    ###
    ### the parameters below are less important
    ###
    n_runs = 1,                 # number of classification passes per text
    n_attributes_per_run = 8,   # if more than 8 labels, will split into separate calls of <= 8
    n_parallels = 3,          # max parallel threads (reduce if many errors, increase for higher speed)
    reset_files = False, # rerunning the cell loads from save / picks up from checkpoint
)

In [ ]:
#I then cleaned these results later
import pandas as pd

blame_df = pd.read_csv("/content/thesis_blame_classify_dedup.csv")

In [ ]:
#Check how many rows have nulls for the article text column
blame_df[blame_df["text"].isna()]

In [ ]:
#Remove rows where the article text is null and article text duplicates
blame_df = blame_df.dropna(subset=["text"]).drop_duplicates(subset=["text"]).copy()
#Check if successful
blame_df[blame_df["text"].isna()]

In [ ]:
#Reset id and index of the dataframe so articles are correctly numbered after removals
blame_df["id"] = range(1, len(blame_df) + 1)
blame_df = blame_df.reset_index(drop=True)

#Confirm articles are numbered correctly
blame_df.tail(5)

In [ ]:
#Extract all articles that failed to classify as either true or false and instead classified as "null", copying them into their own separate dataframe so I can rerun the classification prompt

null_blame = blame_df[blame_df["blames US"].isna()].copy()

print(len(null_blame))

In [ ]:
#Remove the existing classification columns that were transferred from the old dataframe
null_blame = null_blame.drop(columns=["blames US", "predicted_classes"], errors="ignore")

In [ ]:
#Rerun the prompt on these articles themselves
labels = {
    "blames US": "The text attributes blame for the Global Financial Crisis to the US or describes the Global Financial Crisis as the result of the US subprime mortgage crisis, Lehman Brothers declaring bankruptcy, or other references to a crisis in the US that spread."
}

results = await gabriel.classify(
    null_blame,
    column_name = "text",       # name of column with text to classify
    labels = labels,            # dictionary of label definitions, defined above
    save_dir = "30_classify_blame",  # directory to save results, use a Google Drive folder (e.g. "/content/drive/folder") for permanent storage (see 'Loading your data' section)
    model = "gpt-5.4-mini",     # GPT model used for classification (gpt-5.4 is best, gpt-5.4-nano is cheapest and fastest, gpt-5.4-mini is good balance)
    modality = "text",  # input modality can also be "entity" (for terms like 'apple pie', not full texts), "web" (see web section), "pdf", "image", or "audio" (see multimodal section)
    ###
    ### the parameters below are less important
    ###
    n_runs = 1,                 # number of classification passes per text
    n_attributes_per_run = 8,   # if more than 8 labels, will split into separate calls of <= 8
    n_parallels = 150,          # max parallel threads (reduce if many errors, increase for higher speed)
    reset_files = False, # rerunning the cell loads from save / picks up from checkpoint
)

In [ ]:
#Check results
results

In [ ]:
#Save separate classifications as CSV
null_blame.to_csv("null_blame.csv", index=False)
from google.colab import files
files.download("null_blame.csv")

In [ ]:
#Imported the new classification results for null articles from the runtime file contents
retry_df = pd.read_csv("/content/30_classify_blame/classify_responses_cleaned.csv")

In [ ]:
#Create an ID and classification label lookup to merge back into original results dataframe
retry_map = retry_df.set_index("id")["blames US"]

#Map these new classifications onto the previously null classifications in the original df using article ids
mask = blame_df["blames US"].isna()

blame_df.loc[mask, "blames US"] = (
    blame_df.loc[mask, "id"].map(retry_map)
)

#Confirm that there are no nulls in original df after merging
print("Remaining nulls:", blame_df["blames US"].isna().sum())

In [ ]:
#Confirm articles in new df have correct classifications by checking an article I know is classified as false in the rerun
blame_df[blame_df["id"] == 1891]

In [ ]:
#One last check for total articles (should be 10148) and total nulls (should be 0)
print(len(blame_df))
print(blame_df["blames US"].isna().sum())
blame_df.head()

In [ ]:
#Download new complete blame dataframe
blame_df.to_csv("blame_df_full_results.csv", index=False)

from google.colab import files
files.download("blame_df_full_results.csv")

## "Western decline" and "crisis as opportunity"

Repeat process: Cleaning results

In [ ]:
#This is the original decline and opportunity prompt run on 10,192 articles

labels = {
    "Crisis as Opportunity": "The text describes the Global Financial Crisis as an opportunity for China or as something that has had, or will have, a positive impact on China.",
    "Western Decline": "The text frames the Global Financial Crisis as undermining the legitimacy, credibility, or normative authority of Western countries and/or their economic or political systems (e.g., capitalism, neoliberalism, or Western governance)."

}

results_2 = await gabriel.classify(
    df_articles,
    column_name = "text",       # name of column with text to classify
    labels = labels,            # dictionary of label definitions, defined above
    save_dir = "full_classify_decline",  # directory to save results, use a Google Drive folder (e.g. "/content/drive/folder") for permanent storage (see 'Loading your data' section)
    model = "gpt-5.4-mini",     # GPT model used for classification (gpt-5.4 is best, gpt-5.4-nano is cheapest and fastest, gpt-5.4-mini is good balance)
    modality = "text",  # input modality can also be "entity" (for terms like 'apple pie', not full texts), "web" (see web section), "pdf", "image", or "audio" (see multimodal section)
    ###
    ### the parameters below are less important
    ###
    n_runs = 1,                 # number of classification passes per text
    n_attributes_per_run = 5,   # if more than 8 labels, will split into separate calls of <= 8
    n_parallels = 5,          # max parallel threads (reduce if many errors, increase for higher speed)
    reset_files = False, # rerunning the cell loads from save / picks up from checkpoint
)

In [ ]:
#Notice that there is a discrepancy between unique article texts and total rows
print(len(results_2))
print(results_2["text"].nunique())

In [ ]:
#Drop duplicates
results_clean = results_2.drop_duplicates(subset="text").copy()

In [ ]:
#Confirm no more duplicates and download results as csv
len(results_clean)
results_clean["text"].nunique()

results_clean.to_csv("results_clean.csv", index=False)

from google.colab import files
files.download("results_clean.csv")

In [ ]:
import pandas as pd

decline_df = pd.read_csv("/content/results_clean.csv")

In [ ]:
#Remove article text duplicates and article texts that are empty
decline_df = decline_df.dropna(subset=["text"]).drop_duplicates(subset=["text"]).copy()

#Confirm it worked
decline_df[decline_df["text"].isna()]

In [ ]:
#Fix dataframe id and index after article removal
decline_df["id"] = range(1, len(decline_df) + 1)
decline_df = decline_df.reset_index(drop=True)

#Confirm numbers are correct
decline_df.tail(5)

In [ ]:
#See which rows were null for the classifications
null_rows = decline_df[
    decline_df[["Western Decline", "Crisis as Opportunity"]].isnull().any(axis=1)
]

null_rows
#Manual review shows that all the rows that are null are null for both attributes

In [ ]:
#Since all null rows are null for both, new dataframe is created by copying over any article that is null for its Western Decline classification (it is therefore also null for the other attribute)
null_decline = decline_df[decline_df["Western Decline"].isna()].copy()

#Drop columns so GABRIEL can create new ones without error
null_decline = null_decline.drop(columns=["Western Decline", "Crisis as Opportunity", "predicted_classes"], errors="ignore")

In [ ]:
#Rerun GABRIEL on these articles
labels = {
    "Crisis as Opportunity": "The text describes the Global Financial Crisis as an opportunity for China or as something that has had, or will have, a positive impact on China.",
    "Western Decline": "The text frames the Global Financial Crisis as undermining the legitimacy, credibility, or normative authority of Western countries and/or their economic or political systems (e.g., capitalism, neoliberalism, or Western governance)."

}

results = await gabriel.classify(
    null_decline,
    column_name = "text",       # name of column with text to classify
    labels = labels,            # dictionary of label definitions, defined above
    save_dir = "AMEND_31_decline_classify",  # directory to save results, use a Google Drive folder (e.g. "/content/drive/folder") for permanent storage (see 'Loading your data' section)
    model = "gpt-5.4-mini",     # GPT model used for classification (gpt-5.4 is best, gpt-5.4-nano is cheapest and fastest, gpt-5.4-mini is good balance)
    modality = "text",  # input modality can also be "entity" (for terms like 'apple pie', not full texts), "web" (see web section), "pdf", "image", or "audio" (see multimodal section)
    ###
    ### the parameters below are less important
    ###
    n_runs = 1,                 # number of classification passes per text
    n_attributes_per_run = 5,   # if more than 8 labels, will split into separate calls of <= 8
    n_parallels = 150,          # max parallel threads (reduce if many errors, increase for higher speed)
    reset_files = False, # rerunning the cell loads from save / picks up from checkpoint
)

In [ ]:
#Look at results
results

In [ ]:
#Save new classified results as CSV then reimport
fixed_df = pd.read_csv("/content/AMEND_31_decline_classify/classify_responses_cleaned.csv")

In [ ]:
#Merge new results in fixed_df back into the decline_df dataframe.
merged = decline_df.merge(
    fixed_df[["id", "Western Decline", "Crisis as Opportunity"]],
    on="id",
    how="left",
    suffixes=("", "_new")
)

#If a value exists for the classifications in the original df, keep it, but if it's null, replace with new values
for col in ["Western Decline", "Crisis as Opportunity"]:
    merged[col] = merged[col].combine_first(merged[f"{col}_new"])

merged = merged.drop(columns=["Western Decline_new", "Crisis as Opportunity_new"])

In [ ]:
#Create a new dataframe with the merged results
decline_df = merged

In [ ]:
#Confirm that the classifications of the merged and the rerun line up

results[results["id"] == 1891]

decline_df[decline_df["id"] == 1891]

In [ ]:
#Confirm no nulls
print("Remaining nulls:", decline_df["Western Decline"].isna().sum())
print("Remaining nulls:", decline_df["Crisis as Opportunity"].isna().sum())

In [ ]:
#Confirm no nulls and that articles are numbered correctly

print(len(decline_df))
print(decline_df["Western Decline"].isna().sum())  # should be 0
decline_df.tail()

In [ ]:
#Save complete dataframe with corrected results
decline_df.to_csv("decline_df_full_results.csv", index=False)

from google.colab import files
files.download("decline_df_full_results.csv")

GFC Positivity, Reordering, Tone

In [ ]:
import pandas as pd

# You can load real data using df = pd.read_csv("path_to_your_spreadsheet"). See the 'Loading your data' section for more info

# Define the features we want to measure
attributes = {
    "Reordering": "The text uses the Global Financial Crisis to justify changes to the international order, such as reform of global governance and increased role for China (0 = no such argument, 100 = highly emphasized argument or central to the piece).",
    "GFC Positivity": "The text describes China's handling of or response to the Global Financial Crisis as effective, including framing the crisis as a test that China passed (0 = no such framing, 100 = strong emphasis on China’s successful performance).",
    "Tone": "the tone or sentiment of the article (0 = extremely negative or pessimistic, 50 = neutral/mixed, 100 = extremely positive or optimistic)"
}

additional_instructions = "" #### Any additional instructions you want to pass to the model when rating. For instance examples of how to rate the text

# Make the call to the package
results = await gabriel.rate(
    df = df_articles_3,      # if using real data, substitute 'toy_data' with 'df'
    column_name = "text",       # name of column with text to rate
    attributes = attributes,    # attributes to score on 0–100 scale; defined above
    save_dir = "articles_rate_1",      # directory to save results, use a Google Drive folder (e.g. "/content/drive/folder") for permanent storage (see 'Loading your data' section)
    model = "gpt-5.4-mini",     # GPT model used for ratings (gpt-5.4 is best, gpt-5.4-nano is cheapest and fastest, gpt-5.4-mini is good balance)
    modality = "text",  # input modality can also be "entity" (for terms like 'apple pie', not full texts), "web" (see web section), "pdf", "image", or "audio" (see multimodal section)
    ###
    ### the parameters below are less important
    ###
    additional_instructions = additional_instructions,
    n_runs = 1,                 # number of rating passes per text (higher = averaged over more ratings)
    n_attributes_per_run = 8,   # if more than 8 attributes, will split into separate calls of <= 8
    n_parallels = 30,          # max parallel threads (reduce if many errors, increase for higher speed)
    reasoning_effort = 'low',   # amount of reasoning effort, 'none' is default
    reset_files = False  # if False, rerunning the cell loads from save / picks up from checkpoint
)

# Reordering Etc New Classifications

In [ ]:
##Run new classifications

labels = {
      "Reordering": "The text uses the Global Financial Crisis to justify or advocate for changes to the international order, such as reform of global governance and increased role for China.",
      "Pro-China": "The article frames the crisis as evidence of China’s strength, resilience, or historic rejuvenation.",
      "Reform": "The article advocates for reform of international institutions.",
      "Coop Non-West": "The article advocates for increased or continued cooperation with non-Western or developing countries.",
      "Coop West": "The article advocates for increased or continued cooperation with non-Western or developed countries.",
      "China Cred": "The text frames the Global Financial Crisis as demonstrating the legitimacy, credibility, or normative authority of China and/or its economic or political system (e.g., socialism, communism, non-democratic governance)."
}

results = await gabriel.classify(
    df_articles_3,
    column_name = "text",       # name of column with text to classify
    labels = labels,            # dictionary of label definitions, defined above
    save_dir = "full_classify_reordering",  # directory to save results, use a Google Drive folder (e.g. "/content/drive/folder") for permanent storage (see 'Loading your data' section)
    model = "gpt-5.4-mini",     # GPT model used for classification (gpt-5.4 is best, gpt-5.4-nano is cheapest and fastest, gpt-5.4-mini is good balance)
    modality = "text",  # input modality can also be "entity" (for terms like 'apple pie', not full texts), "web" (see web section), "pdf", "image", or "audio" (see multimodal section)
    ###
    ### the parameters below are less important
    ###
    n_runs = 1,                 # number of classification passes per text
    n_attributes_per_run = 8,   # if more than 8 labels, will split into separate calls of <= 8
    n_parallels = 40,          # max parallel threads (reduce if many errors, increase for higher speed)
    reset_files = False, # rerunning the cell loads from save / picks up from checkpoint
)

In [ ]:
import pandas as pd

# create year column from date column
results["year"] = pd.to_datetime(results["date"]).dt.year

# preview
results[["date", "year"]].head()

In [ ]:
# Show rows where either column is null
df_reordering[
    df_reordering["Reordering"].isna() |
    df_reordering["Coop Non-West"].isna()
]

In [ ]:
df_reordering[["Reordering", "Coop Non-West"]].isna().sum()

In [ ]:
# Create a new dataframe with any rows where either column is null
df_nulls = df_reordering[
    df_reordering["Reordering"].isna() |
    df_reordering["Coop Non-West"].isna()
].copy()

In [ ]:
#Check columns
df_nulls

In [ ]:
# Keep only selected columns
df_nulls = df_nulls[["author", "date", "title", "id", "text"]].copy()
df_nulls.head(5)

In [ ]:
#rerunning on null articles
labels = {
      "Reordering": "The text uses the Global Financial Crisis to justify or advocate for changes to the international order, such as reform of global governance and increased role for China.",
      "Pro-China": "The article frames the crisis as evidence of China’s strength, resilience, or historic rejuvenation.",
      "Reform": "The article advocates for reform of international institutions.",
      "Coop Non-West": "The article advocates for increased or continued cooperation with non-Western or developing countries.",
      "Coop West": "The article advocates for increased or continued cooperation with non-Western or developed countries.",
      "China Cred": "The text frames the Global Financial Crisis as demonstrating the legitimacy, credibility, or normative authority of China and/or its economic or political system (e.g., socialism, communism, non-democratic governance)."
}

results = await gabriel.classify(
    df_nulls,
    column_name = "text",       # name of column with text to classify
    labels = labels,            # dictionary of label definitions, defined above
    save_dir = "rerun_classify",  # directory to save results, use a Google Drive folder (e.g. "/content/drive/folder") for permanent storage (see 'Loading your data' section)
    model = "gpt-5.4-mini",     # GPT model used for classification (gpt-5.4 is best, gpt-5.4-nano is cheapest and fastest, gpt-5.4-mini is good balance)
    modality = "text",  # input modality can also be "entity" (for terms like 'apple pie', not full texts), "web" (see web section), "pdf", "image", or "audio" (see multimodal section)
    ###
    ### the parameters below are less important
    ###
    n_runs = 1,                 # number of classification passes per text
    n_attributes_per_run = 8,   # if more than 8 labels, will split into separate calls of <= 8
    n_parallels = 40,          # max parallel threads (reduce if many errors, increase for higher speed)
    reset_files = False, # rerunning the cell loads from save / picks up from checkpoint
)

In [ ]:
results

In [ ]:
#merge results classifications for "Coop Non-West" and "Reordering" to df_reordering_5
class_cols = ["Reordering", "Pro-China", "Reform", "Coop Non-West", "Coop West", "China Cred"]

# make sure article IDs match by type
df_reordering_5["id"] = df_reordering_5["id"].astype("Int64")
results["id"] = results["id"].astype("Int64")

# rows in df_reordering_5 where Reordering OR Coop Non-West is null
null_mask = (
    df_reordering_5["Reordering"].isna() |
    df_reordering_5["Coop Non-West"].isna()
)

# use article id as lookup key
results_lookup = results.set_index("id")

# only update rows whose ids exist in results
ids_to_update = df_reordering_5.loc[null_mask, "id"]
ids_to_update = ids_to_update[ids_to_update.isin(results_lookup.index)]

# move classifications from results into df_reordering_5
for article_id in ids_to_update:
    df_reordering_5.loc[
        df_reordering_5["id"] == article_id,
        class_cols
    ] = results_lookup.loc[article_id, class_cols].values

# check remaining nulls in classification columns
df_reordering_5[class_cols].isna().sum()

In [ ]:
#Confirm merging worked
df_reordering_5[df_reordering_5["id"] == 882][["id"] + class_cols]

In [ ]:
# Null count in each column
df_reordering_5.isna().sum()

In [ ]:
# Save dataframe to CSV in Colab
df_reordering_5.to_csv("df_reordering_no_nulls.csv", index=False)

#download
from google.colab import files
files.download("df_reordering_no_nulls.csv")

# Checking Results: Classifications

##Crisis as Opportunity Results

In [ ]:
#Results for cleaned "Western Decline" and "Crisis as Opportunity"
import pandas as pd

new_dec_opp_df = pd.read_csv("/content/Decline_Opp_Classified-no-nulls.csv")

# Create yearly summary
crisis_summary = (
    new_dec_opp_df
    .groupby("year")
    .agg(
        total_articles=("Crisis as Opportunity", "count"),
        true_articles=("Crisis as Opportunity", lambda x: (x == True).sum())
    )
    .reset_index()
)

# Compute percentage
crisis_summary["percent_true"] = (
    crisis_summary["true_articles"] / crisis_summary["total_articles"] * 100
)

# Create totals row
crisis_totals = pd.DataFrame({
    "year": ["Total"],
    "total_articles": [crisis_summary["total_articles"].sum()],
    "true_articles": [crisis_summary["true_articles"].sum()]
})

crisis_totals["percent_true"] = (
    crisis_totals["true_articles"] / crisis_totals["total_articles"] * 100
)

# Append totals row
crisis_summary_with_total = pd.concat([crisis_summary, crisis_totals], ignore_index=True)

# Optional: round percentages
crisis_summary_with_total["percent_true"] = crisis_summary_with_total["percent_true"].round(2)

# View result
crisis_summary_with_total

In [ ]:
import matplotlib.pyplot as plt

# --- Crisis as Opportunity line graph ---
crisis_plot = crisis_summary_with_total[
    crisis_summary_with_total["year"] != "Total"
].copy()

crisis_plot["year"] = crisis_plot["year"].astype(int)

plt.figure(figsize=(10,6))

# line + dots
plt.plot(
    crisis_plot["year"],
    crisis_plot["percent_true"],
    marker="o",
    linewidth=2
)

plt.title("Crisis as Opportunity Over Time")
plt.xlabel("Year")
plt.ylabel("Percent of Articles Classified True")

# labels above each point
for x, y in zip(crisis_plot["year"], crisis_plot["percent_true"]):
    plt.text(
        x,
        y + 2,
        f"{y:.1f}",
        ha="center",
        va="bottom",
        fontsize=10
    )

plt.xticks(crisis_plot["year"])
plt.ylim(0, 100)

plt.tight_layout()
plt.show()

In [ ]:
#Reworked line graph
# Plot
plt.figure(figsize=(10,6))
plt.plot(summary.index, summary.values, marker='o', linewidth=2, label="Crisis as Opportunity")

# Add value labels above each dot
for x, y in zip(summary.index, summary.values):
    plt.text(x, y + 2, f"{y:.1f}", ha='center', fontsize=10)

# Labels
plt.title("The GFC as Opportunity", fontsize=14)
plt.xlabel("Year", fontsize=12)
plt.ylabel("Percent of Articles True", fontsize=12)

# Formatting
plt.xticks(summary.index, rotation=45)
plt.yticks(range(0, 101, 10))
plt.ylim(0, 100)

# No grid
# plt.grid(...) removed

plt.tight_layout()
plt.show()

In [ ]:
# Total percentage of all articles classified True for Crisis as Opportunity
overall_pct = (df_decline_opp["Crisis as Opportunity"] == True).mean() * 100

print(f"Overall % of articles True for Crisis as Opportunity: {overall_pct:.2f}%")

##Western Decline Results

In [ ]:
import pandas as pd

new_dec_opp_df = pd.read_csv("/content/Decline_Opp_Classified-no-nulls.csv")

In [ ]:
new_dec_opp_df

In [ ]:
# Create yearly summary for Western Decline
western_summary = (
    new_dec_opp_df
    .groupby("year")
    .agg(
        total_articles=("Western Decline", "count"),
        true_articles=("Western Decline", lambda x: (x == True).sum())
    )
    .reset_index()
)

# Compute percentage
western_summary["percent_true"] = (
    western_summary["true_articles"] / western_summary["total_articles"] * 100
)

# Create totals row
western_totals = pd.DataFrame({
    "year": ["Total"],
    "total_articles": [western_summary["total_articles"].sum()],
    "true_articles": [western_summary["true_articles"].sum()]
})

western_totals["percent_true"] = (
    western_totals["true_articles"] / western_totals["total_articles"] * 100
)

# Append totals row
western_summary_with_total = pd.concat([western_summary, western_totals], ignore_index=True)

# Optional: round percentages
western_summary_with_total["percent_true"] = western_summary_with_total["percent_true"].round(2)

# View result
western_summary_with_total

In [ ]:
import matplotlib.pyplot as plt

# 1) Calculate yearly % True for Western Decline
summary = (
    new_dec_opp_df.groupby("year")["Western Decline"]
    .mean()              # True = 1, False = 0
    .mul(100)            # convert to percentage
    .reset_index(name="percentage")
)

# 2) Visualize as line chart
plt.figure(figsize=(10,6))

plt.plot(
    summary["year"],
    summary["percentage"],
    marker="o",
    linewidth=2
)

# add labels above each point
for x, y in zip(summary["year"], summary["percentage"]):
    plt.text(x, y + 0.8, f"{y:.1f}", ha="center", fontsize=10)

plt.xlabel("Year")
plt.ylabel("Percent of Articles Showing Western Decline")
plt.title("Western Decline Over Time")
plt.ylim(0, 100)

plt.tight_layout()
plt.show()

##Total Articles by Year

In [ ]:
import pandas as pd

df = pd.read_csv("/content/Blame-Classified-no-nulls-Complete.csv")

In [ ]:
#Organize dates into a separate year column
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year

year_counts = df[df["blames US"] == True].groupby("year").size()

In [ ]:
#look at results by year
summary1 = df.groupby("year")["blames US"].agg(
    total_articles="count",
    blames_us_true="sum"
).reset_index()

summary1

In [ ]:
#Create percentage values for each year

summary1["percent_true"] = (
    summary1["blames_us_true"] / summary1["total_articles"] * 100
).round(2)

totals_row = pd.DataFrame([{
    "year": "Total",
    "total_articles": summary1["total_articles"].sum(),
    "blames_us_true": summary1["blames_us_true"].sum(),
    "percent_true": round(
        summary1["blames_us_true"].sum() / summary1["total_articles"].sum() * 100,
        2
    )
}])

summary_with_total = pd.concat([summary, totals_row], ignore_index=True)

summary_with_total

In [ ]:
#Create total articles per year graph
import matplotlib.pyplot as plt

plt.figure(figsize=(12,7))

plt.plot(
    summary['year'],
    summary['total_articles'],
    marker='o',
    linewidth=2.5,
    markersize=8
)

offset = summary['total_articles'].max() * 0.03

for x, y in zip(summary['year'], summary['total_articles']):
    plt.text(
    x - 0.08,
    y - 40,
    f"{int(y):,}",
    ha='right',
    va='top',
    fontsize=11
)

plt.xlabel("Year", fontsize=13)
plt.ylabel("Number of Articles", fontsize=13)
plt.title("Total Articles Published Per Year", fontsize=18, pad=15)

plt.xticks(fontsize=11)
plt.yticks(fontsize=11)

plt.tight_layout()
plt.show()

##Blame US Results

In [ ]:
#Create blames US line graph
import matplotlib.pyplot as plt

# yearly percentages
summary1["percentage"] = (summary1["blames_us_true"] / summary1["total_articles"]) * 100

plt.figure(figsize=(10,6))

# line with dots
plt.plot(summary1["year"], summary1["percentage"], marker='o', linewidth=2)

# add labels next to each dot
for x, y in zip(summary1["year"], summary1["percentage"]):
    plt.text(x, y + 1, f"{y:.1f}", ha='center', fontsize=10)

plt.xlabel("Year")
plt.ylabel("Percent of Articles Blaming US")
plt.title("Articles Blaming the US Over Time")

plt.ylim(0, 100)

plt.tight_layout()
plt.show()

##Create Combined Panel of Blames US and Western Decline Graph

In [ ]:
import matplotlib.pyplot as plt

# --- prepare data ---
summary1 = summary1.copy()   # blames US dataframe
summary = summary.copy()   # replace with your western decline summary if separate

# Example:
# summary1 = summary_blames_us
# summary2 = summary_western_decline

fig, axes = plt.subplots(1, 2, figsize=(10,4), sharey=True)

# ---------------- LEFT GRAPH: Blames US ----------------
axes[0].plot(summary1["year"], summary1["percent_true"], marker="o", linewidth=2)

for x, y in zip(summary1["year"], summary1["percent_true"]):
    axes[0].text(x, y + 2, f"{y:.1f}", ha="center", fontsize=10)

axes[0].set_title("Articles Blaming the US Over Time")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Percent of Articles")
axes[0].set_ylim(0, 100)

# ---------------- RIGHT GRAPH: Western Decline ----------------
axes[1].plot(summary["year"], summary["percentage"], marker="o", linewidth=2)

for x, y in zip(summary["year"], summary["percentage"]):
    axes[1].text(x, y + 2, f"{y:.1f}", ha="center", fontsize=10)

axes[1].set_title("Western Decline Over Time")
axes[1].set_xlabel("Year")
axes[1].set_ylim(0, 100)

# ---------------- FINAL TOUCHES ----------------
plt.tight_layout()
plt.show()

Check overlap between blames us and western decline

In [ ]:
##Robustness Check

import pandas as pd

df_blame = pd.read_csv("/content/Blame-Classified-no-nulls-Complete.csv")
df_decline = pd.read_csv("/content/Decline_Opp_Classified-no-nulls.csv")

In [ ]:
#Combine the two dfs into one
df_combined = df_blame.merge(
    df_decline[["id", "Western Decline"]],
    on="id",
    how="left"
)

In [ ]:
#Check how many articles are true for both
both_true = df_combined[
    (df_combined["blames US"] == True) &
    (df_combined["Western Decline"] == True)
]

len(both_true) / len(df_combined)

In [ ]:
#How many articles that are true for "blames US" are also true for "Western Decline"
blame_true = df_combined[df_combined["blames US"] == True]

(blame_true["Western Decline"] == True).mean()

In [ ]:
#How many articles true for "western decline" are also true for "blames US"
decline_true = df_combined[df_combined["Western Decline"] == True]

(decline_true["blames US"] == True).mean()

##Reordering, reform, etc

In [ ]:
import pandas as pd

df_reordering_2 = pd.read_csv("/content/thesis_df_new_classify_no_nulls.csv")

In [ ]:
df_reordering_2[["Reordering", "Coop Non-West"]].isna().sum()

In [ ]:
#Organize dates into a separate year column
df_reordering["date"] = pd.to_datetime(df_reordering["date"])
df_reordering["year"] = df_reordering["date"].dt.year

In [ ]:
summary_all = (
    df_reordering.groupby("year")[
        ["Reordering", "Pro-China", "Reform",
         "Coop Non-West", "Coop West", "China Cred"]
    ]
    .mean()
    .mul(100)
    .reset_index()
)

summary_all

##New World Order Results

In [ ]:
#Sample of articles classified "true" for reordering in 2015
df_reordering.loc[
    (df_reordering["year"] == 2015) &
    (df_reordering["Reordering"] == True),
    ["date", "title", "text"]
].sample(10, random_state=42)

In [ ]:
#Reordering Line graph

# Create a year column from the date column in classify_results
df_reordering_5["year"] = pd.to_datetime(df_reordering_5["date"]).dt.year

# choose column
col = "Reordering"

# yearly % true
summary1 = (
    df_reordering_5.groupby("year")[col]
    .mean()
    .mul(100)
    .reset_index(name="percentage_true")
)

plt.figure(figsize=(10,6))

plt.plot(
    summary1["year"],
    summary1["percentage_true"],
    marker='o',
    linewidth=2,
    label=col
)

# value labels
for x, y in zip(summary1["year"], summary1["percentage_true"]):
    plt.text(x, y + 2, f"{y:.1f}", ha='center', fontsize=9)

plt.xlabel("Year")
plt.ylabel("Percent of Articles")
plt.title("Advocates Changes to the World Order")

plt.ylim(0,100)
plt.tight_layout()
plt.show()

In [ ]:
# overall percentage of articles classified True for Reordering
percent_true = df_reordering_5["Reordering"].mean() * 100

print(f"Reordering: {percent_true:.2f}% of all articles are True")

##Reform Results

In [ ]:
# Create a year column from the date column in classify_results
classify_results["year"] = pd.to_datetime(classify_results["date"]).dt.year

# choose column
col = "Reform"

# yearly % true
summary = (
    results.groupby("year")[col]
    .mean()
    .mul(100)
    .reset_index(name="percentage_true")
)

plt.figure(figsize=(10,6))

plt.plot(
    summary["year"],
    summary["percentage_true"],
    marker='o',
    linewidth=2,
    label=col
)

# value labels
for x, y in zip(summary["year"], summary["percentage_true"]):
    plt.text(x, y + 1, f"{y:.1f}", ha='center', fontsize=9)

plt.xlabel("Year")
plt.ylabel("Percent of Articles True")
plt.title("Reform of International Institutions")

plt.ylim(0,30)
plt.tight_layout()
plt.show()

##GFC as evidence of strength, resilience

In [ ]:
# Create a year column from the date column in classify_results
classify_results["year"] = pd.to_datetime(classify_results["date"]).dt.year

#choose column
col = "Pro-China"

# yearly % true
summary = (
    classify_results.groupby("year")[col]
    .mean()
    .mul(100)
    .reset_index(name="percentage_true")
)

plt.figure(figsize=(10,6))

plt.plot(
    summary["year"],
    summary["percentage_true"],
    marker='o',
    linewidth=2,
    label=col
)

# value labels
for x, y in zip(summary["year"], summary["percentage_true"]):
    plt.text(x, y + 1, f"{y:.1f}", ha='center', fontsize=9)

plt.xlabel("Year")
plt.ylabel("Percent of Articles True")
plt.title("GFC as Evidence of Chinese Strength, Resilience")

plt.ylim(0,80)
plt.tight_layout()
plt.show()

In [ ]:
# overall percentage of articles classified True for Pro-China (Strength, resilience)
percent_true = classify_results["Pro-China"].mean() * 100

print(f"Pro-China: {percent_true:.2f}% of all articles are True")

##Cooperation with non-Western countries

In [ ]:
df_reordering

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# create year column if needed
df_reordering_5["year"] = pd.to_datetime(df_reordering_5["date"]).dt.year

# choose column
col = "Coop Non-West"

# yearly % true
summary = (
    df_reordering_5.groupby("year")[col]
    .mean()
    .mul(100)
    .reset_index(name="percentage_true")
)

plt.figure(figsize=(10,6))

plt.plot(
    summary["year"],
    summary["percentage_true"],
    marker='o',
    linewidth=2,
    label=col
)

# value labels
for x, y in zip(summary["year"], summary["percentage_true"]):
    plt.text(x, y + 2, f"{y:.1f}", ha='center', fontsize=9)

plt.xlabel("Year")
plt.ylabel("Percent of Articles")
plt.title("Advocates for Cooperation with Non-Western or Developing Countries")

plt.ylim(0,100)
plt.tight_layout()
plt.show()

In [ ]:
# choose your column
col = "Coop Non-West"   # change this

# overall percentage true in full corpus
percent_true = df_reordering_5[col].mean() * 100

print(f"{col}: {percent_true:.2f}% of all articles are True")

##Visualizing Combined Collaboration Non-West and Reordering

In [ ]:
import matplotlib.pyplot as plt

# ---------------------------
# 1) Calculate yearly % True
# ---------------------------

summary_reordering = (
    df_reordering_5.groupby("year")["Reordering"]
    .mean()
    .mul(100)
    .reset_index(name="percentage")
)

summary_coop = (
    df_reordering_5.groupby("year")["Coop Non-West"]
    .mean()
    .mul(100)
    .reset_index(name="percentage")
)

# ---------------------------
# 2) Create side-by-side panel
# ---------------------------

fig, axes = plt.subplots(1, 2, figsize=(11,5), sharey=True)

# -------- Left: Reordering --------
axes[0].plot(
    summary_reordering["year"],
    summary_reordering["percentage"],
    marker="o",
    linewidth=2
)

for x, y in zip(summary_reordering["year"], summary_reordering["percentage"]):
    axes[0].text(x, y + 2, f"{y:.1f}", ha="center", fontsize=10)

axes[0].set_title("Advocates Changes to the World Order")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Percent of Articles")
axes[0].set_ylim(0, 100)

# -------- Right: Coop Non-West --------
axes[1].plot(
    summary_coop["year"],
    summary_coop["percentage"],
    marker="o",
    linewidth=2
)

for x, y in zip(summary_coop["year"], summary_coop["percentage"]):
    axes[1].text(x, y + 2.8, f"{y:.1f}", ha="center", fontsize=10)

axes[1].set_title("Advocates Cooperation with Non-West")
axes[1].set_xlabel("Year")
axes[1].set_ylim(0, 100)

# ---------------------------
# Final layout
# ---------------------------
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# ---------------------------
# Yearly percentages
# ---------------------------

summary_reordering = (
    df_reordering_5.groupby("year")["Reordering"]
    .mean()
    .mul(100)
    .reset_index(name="percentage")
)

summary_coop = (
    df_reordering_5.groupby("year")["Coop Non-West"]
    .mean()
    .mul(100)
    .reset_index(name="percentage")
)

# ---------------------------
# Single combined graph
# ---------------------------

plt.figure(figsize=(10,6))

# Reordering line
plt.plot(
    summary_reordering["year"],
    summary_reordering["percentage"],
    marker="o",
    linewidth=2,
    label="Advocates Changes to World Order"
)

# Coop line
plt.plot(
    summary_coop["year"],
    summary_coop["percentage"],
    marker="o",
    linewidth=2,
    label="Advocates Cooperation with Non-West"
)

# Point labels
for x, y in zip(summary_reordering["year"], summary_reordering["percentage"]):
    plt.text(x, y + 0.8, f"{y:.1f}", ha="center", fontsize=9)

for x, y in zip(summary_coop["year"], summary_coop["percentage"]):
    plt.text(x, y + 0.8, f"{y:.1f}", ha="center", fontsize=9)

# Reset to Matplotlib default font/style
plt.rcdefaults()

plt.xlabel("Year")
plt.ylabel("Percent of Articles")
plt.title("Adocating for a New World Order and Non-West Cooperation")
plt.ylim(0, 100)
plt.legend()
plt.tight_layout()
plt.show()

# Ratings

##First Set of Ratings

In [ ]:
import pandas as pd

# You can load real data using df = pd.read_csv("path_to_your_spreadsheet"). See the 'Loading your data' section for more info

# Define the features we want to measure
attributes = {
    "Crisis Response Attitude": "The text describes the response of China, including the Chinese government, economy, firms, and society, to the Global Financial Crisis (0 = extremely negative or pessimistic, 50 = neutral/mixed, 100 = extremely positive or optimistic).",
    "Crisis Attitude": "The text describes the impact of the Global Financial Crisis on China, including the Chinese government, economy, firms, and society (0 = extremely negative or pessimistic, 50 = neutral/mixed, 100 = extremely positive or optimistic).",
    "Coop vs. Comp: Overall": "The text describes the role of other countries in China's response to the Global Financial Crisis or how the Global Financial Crisis impacted relations with foreign countries (0 = extremely negative or adversarial, 50 = mixed or neutral, 100 = extremely positive or cooperative).",
    "Coop vs. Comp: West": "The text describes the role of Western developed countries in China's response to the Global Finanical Crisis or how the Global Financial Crisis impacted or will impact relations with Western developed countries (0 = extremely negative or adversarial, 50 = mixed or neutral, 100 = extremely positive or cooperative).",
    "Coop vs. Comp: Non-West": "The text describes the role of non-Western or developing countries in China's response to the Crisis or how the Crisis impacted relations with non-Western or developing countries (0 = extremely negative or adversarial, 50 = mixed or neutral, 100 = extremely positive or cooperative).",
    "Attitudes West": "The attitude of the text towards Western countries or developed countries (0 = very negative, 50 = neutral or not mentioned, 100 = extremely positive).",
    "Attitudes Non-West": "The attitude of the text towards non-Western countries or developing countries (0 = very negative, 50 = neutral or not mentioned, 100 = extremely positive).",
    "Tone": "The overall sentiment or tone of the article (0 = extremely negative or pessimistic, 50 = neutral/mixed, 100 = extremely positive or optimistic)."
}

additional_instructions = "" #### Any additional instructions you want to pass to the model when rating. For instance examples of how to rate the text

# Make the call to the package
results = await gabriel.rate(
    df = df_articles_3,      # if using real data, substitute 'toy_data' with 'df'
    column_name = "text",       # name of column with text to rate
    attributes = attributes,    # attributes to score on 0–100 scale; defined above
    save_dir = "articles_rate_1",      # directory to save results, use a Google Drive folder (e.g. "/content/drive/folder") for permanent storage (see 'Loading your data' section)
    model = "gpt-5.4-mini",     # GPT model used for ratings (gpt-5.4 is best, gpt-5.4-nano is cheapest and fastest, gpt-5.4-mini is good balance)
    modality = "text",  # input modality can also be "entity" (for terms like 'apple pie', not full texts), "web" (see web section), "pdf", "image", or "audio" (see multimodal section)
    ###
    ### the parameters below are less important
    ###
    additional_instructions = additional_instructions,
    n_runs = 1,                 # number of rating passes per text (higher = averaged over more ratings)
    n_attributes_per_run = 8,   # if more than 8 attributes, will split into separate calls of <= 8
    n_parallels = 30,          # max parallel threads (reduce if many errors, increase for higher speed)
    reasoning_effort = 'low',   # amount of reasoning effort, 'none' is default
    reset_files = False  # if False, rerunning the cell loads from save / picks up from checkpoint
)

##Second Set of Ratings

In [ ]:
import pandas as pd

# You can load real data using df = pd.read_csv("path_to_your_spreadsheet"). See the 'Loading your data' section for more info

# Define the features we want to measure
attributes = {
    "China's Significance": "The text describes China's significance or importance with respect to other countries or in global affairs broadly (0 = not portrayed as significant, 50 = moderately or implicitly significant, 100 = extremely powerful or central).",
    "US Significance": "The text describes the significance or importance of the United States with respect to other countries or in global affairs broadly (0 = not portrayed as significant, 50 = moderately or implicitly significant, 100 = extremely powerful or central).",
    "Attitudes Capitalism": "The attitude of the text toward capitalism or market-oriented economic systems (0 = strongly critical, 50 = neutral or mixed, 100 = strongly supportive).",
    "Attitudes Socialism": "The attitude of the text toward socialism, state-led development, or alternative economic models to capitalism (0 = strongly critical, 50 = neutral or mixed, 100 = strongly supportive).",
    "Coop vs. Comp: Non-West": "The text describes the role of non-Western or developing countries in China's response to the Global Financial Crisis or how the Global Financial Crisis impacted relations with non-Western or developing countries (0 = extremely negative or adversarial, 50 = mixed or neutral, 100 = extremely positive or cooperative).",
    "Focus": "The extent to which the article focuses on China's domestic affairs versus international or foreign affairs (0 = entirely focused on foreign countries or international affairs, 50 = mixed, 100 = entirely focused on China's domestic affairs)."
}

additional_instructions = "" #### Any additional instructions you want to pass to the model when rating. For instance examples of how to rate the text

# Make the call to the package
results_2 = await gabriel.rate(
    df = df_articles_3,      # if using real data, substitute 'toy_data' with 'df'
    column_name = "text",       # name of column with text to rate
    attributes = attributes,    # attributes to score on 0–100 scale; defined above
    save_dir = "articles_rate_2",      # directory to save results, use a Google Drive folder (e.g. "/content/drive/folder") for permanent storage (see 'Loading your data' section)
    model = "gpt-5.4-mini",     # GPT model used for ratings (gpt-5.4 is best, gpt-5.4-nano is cheapest and fastest, gpt-5.4-mini is good balance)
    modality = "text",  # input modality can also be "entity" (for terms like 'apple pie', not full texts), "web" (see web section), "pdf", "image", or "audio" (see multimodal section)
    ###
    ### the parameters below are less important
    ###
    additional_instructions = additional_instructions,
    n_runs = 1,                 # number of rating passes per text (higher = averaged over more ratings)
    n_attributes_per_run = 8,   # if more than 8 attributes, will split into separate calls of <= 8
    n_parallels = 30,          # max parallel threads (reduce if many errors, increase for higher speed)
    reasoning_effort = 'low',   # amount of reasoning effort, 'none' is default
    reset_files = False  # if False, rerunning the cell loads from save / picks up from checkpoint
)

In [ ]:
import pandas as pd

# You can load real data using df = pd.read_csv("path_to_your_spreadsheet"). See the 'Loading your data' section for more info

# Define the features we want to measure
attributes = {
    "Crisis Response Attitude": "The text describes the response of China, including the Chinese government, economy, firms, and society, to the Global Financial Crisis (0 = extremely negative or pessimistic, 50 = neutral/mixed, 100 = extremely positive or optimistic).",
    "Crisis Attitude": "The text describes the impact of the Global Financial Crisis on China, including the Chinese government, economy, firms, and society (0 = extremely negative or pessimistic, 50 = neutral/mixed, 100 = extremely positive or optimistic).",
    "Coop vs. Comp: Overall": "The text describes the role of other countries in China's response to the Global Financial Crisis or how the Global Financial Crisis impacted relations with foreign countries (0 = extremely negative or adversarial, 50 = mixed or neutral, 100 = extremely positive or cooperative).",
    "Coop vs. Comp: West": "The text describes the role of Western developed countries in China's response to the Global Finanical Crisis or how the Global Financial Crisis impacted or will impact relations with Western developed countries (0 = extremely negative or adversarial, 50 = mixed or neutral, 100 = extremely positive or cooperative).",
    "Coop vs. Comp: Non-West": "The text describes the role of non-Western or developing countries in China's response to the Crisis or how the Crisis impacted relations with non-Western or developing countries (0 = extremely negative or adversarial, 50 = mixed or neutral, 100 = extremely positive or cooperative).",
    "Attitudes West": "The attitude of the text towards Western countries or developed countries (0 = very negative, 50 = neutral or not mentioned, 100 = extremely positive).",
    "Attitudes Non-West": "The attitude of the text towards non-Western countries or developing countries (0 = very negative, 50 = neutral or not mentioned, 100 = extremely positive).",
    "Tone": "The overall sentiment or tone of the article (0 = extremely negative or pessimistic, 50 = neutral/mixed, 100 = extremely positive or optimistic)."
}

additional_instructions = "" #### Any additional instructions you want to pass to the model when rating. For instance examples of how to rate the text

# Make the call to the package
results = await gabriel.rate(
    df = df_articles_3,      # if using real data, substitute 'toy_data' with 'df'
    column_name = "text",       # name of column with text to rate
    attributes = attributes,    # attributes to score on 0–100 scale; defined above
    save_dir = "articles_rate_1",      # directory to save results, use a Google Drive folder (e.g. "/content/drive/folder") for permanent storage (see 'Loading your data' section)
    model = "gpt-5.4-mini",     # GPT model used for ratings (gpt-5.4 is best, gpt-5.4-nano is cheapest and fastest, gpt-5.4-mini is good balance)
    modality = "text",  # input modality can also be "entity" (for terms like 'apple pie', not full texts), "web" (see web section), "pdf", "image", or "audio" (see multimodal section)
    ###
    ### the parameters below are less important
    ###
    additional_instructions = additional_instructions,
    n_runs = 1,                 # number of rating passes per text (higher = averaged over more ratings)
    n_attributes_per_run = 8,   # if more than 8 attributes, will split into separate calls of <= 8
    n_parallels = 30,          # max parallel threads (reduce if many errors, increase for higher speed)
    reasoning_effort = 'low',   # amount of reasoning effort, 'none' is default
    reset_files = False  # if False, rerunning the cell loads from save / picks up from checkpoint
)

# Checking Results: Ratings

##Western Failure

In [ ]:
import matplotlib.pyplot as plt

#year counts
df_failure["date"] = pd.to_datetime(df_failure["date"])
df_failure["year"] = df_failure["date"].dt.year

# yearly average rating for Western Failure
summary = (
    df_failure.groupby("year")["Western Failure"]
    .mean()
    .reset_index(name="average_rating")
)

plt.figure(figsize=(10,6))

# line with dots
plt.plot(summary["year"], summary["average_rating"], marker='o', linewidth=2)

# labels next to each dot
for x, y in zip(summary["year"], summary["average_rating"]):
    plt.text(x, y + 1, f"{y:.1f}", ha='center', fontsize=10)

plt.xlabel("Year")
plt.ylabel("Average Western Failure Rating (0–100)")
plt.title("Average Western Failure Rating Over Time")

plt.ylim(0, 30)  # keeps scale consistent with rating system
plt.tight_layout()
plt.show()

##China vs. US Significance

In [ ]:
df_rate9["date"] = pd.to_datetime(df_rate9["date"])
df_rate9["year"] = df_rate9["date"].dt.year

#Mean of ratings for articles each year
yearly_sig = df_rate9.groupby('year')[
    ["China's Significance", "US Significance"]
].mean().reset_index()

In [ ]:
df_rate9[df_rate9["id"] == 9952]

In [ ]:
df_rate9.loc[
    (df_rate9["year"] == 2015) &
    (df_rate9["US Significance"] > 50),
    ["date", "title", "US Significance", "text"]
].sample(10, random_state=42)

In [ ]:
df_rate9[df_rate9["US Significance"] > 50].head(10)

In [ ]:
df_rate9[df_rate9["China's Significance"] > 50].head(10)

In [ ]:
#Graph of China vs. US Significance
import pandas as pd
import matplotlib.pyplot as plt

# create year column from date if needed
df_rate9["year"] = pd.to_datetime(df_rate9["date"]).dt.year

# yearly averages from df_rate9
yearly_sig = (
    df_rate9.groupby("year")[["China's Significance", "US Significance"]]
    .mean()
    .reset_index()
)

plt.figure(figsize=(10,6))

# China line
plt.plot(
    yearly_sig["year"],
    yearly_sig["China's Significance"],
    marker='o',
    linewidth=2,
    label="China's Significance"
)

# US line
plt.plot(
    yearly_sig["year"],
    yearly_sig["US Significance"],
    marker='o',
    linewidth=2,
    label="US Significance"
)

# value labels
for x, y in zip(yearly_sig["year"], yearly_sig["China's Significance"]):
    plt.text(x, y + 1, f"{y:.1f}", ha='center', fontsize=9)

for x, y in zip(yearly_sig["year"], yearly_sig["US Significance"]):
    plt.text(x, y - 3, f"{y:.1f}", ha='center', fontsize=9)

# neutral line
plt.axhline(
    y=50,
    linestyle='--',
    linewidth=1.5,
    color='gray',
    label="Neutral"
)

plt.xlabel("Year")
plt.ylabel("Average Rating")
plt.title("Significance of China vs. US")
plt.ylim(0,100)

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# overall average rating for China and US significance across all years

avg_us = df_rate9["US Significance"].mean()
avg_china = df_rate9["China's Significance"].mean()

print(f"Average US Significance: {avg_us:.2f}")
print(f"Average China's Significance: {avg_china:.2f}")

##Domestic vs. International Focus

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# create year column if needed
df_rate9["year"] = pd.to_datetime(df_rate9["date"]).dt.year

# yearly averages
summary = (
    df_rate9.groupby("year")["Focus"]
    .mean()
    .reset_index(name="average_rating")
)

plt.figure(figsize=(10,6))

plt.plot(
    summary["year"],
    summary["average_rating"],
    marker='o',
    linewidth=2,
    label="Focus"
)

# value labels
for x, y in zip(summary["year"], summary["average_rating"]):
    plt.text(x, y + 3, f"{y:.1f}", ha='center', fontsize=9)

# neutral line
plt.axhline(
    y=50,
    linestyle='--',
    linewidth=1.5,
    color='gray',
    label="Mixed (50)"
)

plt.xlabel("Year")
plt.ylabel("Average Focus Rating (0 = International, 100 = Domestic)")
plt.title("Domestic vs International Focus Over Time")

plt.ylim(0,100)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# overall average Focus rating
avg_focus = df_rate9["Focus"].mean()

print(f"Average Focus rating: {avg_focus:.2f}")

##Attitudes West vs. Non-West

In [ ]:
#import matplotlib.pyplot as plt

# yearly averages for both columns
summary = (
    df_rate8.groupby("year")[["Attitudes West", "Attitudes Non-West"]]
    .mean()
    .reset_index()
)

plt.figure(figsize=(10,6))

# line 1: West
plt.plot(
    summary["year"],
    summary["Attitudes West"],
    marker='o',
    linewidth=2,
    label="West"
)

# line 2: Non-West
plt.plot(
    summary["year"],
    summary["Attitudes Non-West"],
    marker='o',
    linewidth=2,
    label="Non-West"
)
# optional value labels
for x, y in zip(summary["year"], summary["Attitudes West"]):
    plt.text(x, y + 1, f"{y:.1f}", ha='center', fontsize=9)

for x, y in zip(summary["year"], summary["Attitudes Non-West"]):
    plt.text(x, y - 3, f"{y:.1f}", ha='center', fontsize=9)

  # dotted neutral line at 50
plt.axhline(
    y=50,
    linestyle='--',
    linewidth=1.5,
    color='gray',
    label="Neutral or mixed"
)

plt.xlabel("Year")
plt.ylabel("Average Rating")
plt.title("Attitudes About Foreign Countries")
plt.ylim(45,75)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# calculate overall averages for both columns for attitudes
overall_west = df_rate8["Attitudes West"].mean()
overall_nonwest = df_rate8["Attitudes Non-West"].mean()

print("Overall Average - Attitudes West:", round(overall_west, 2))
print("Overall Average - Attitudes Non-West:", round(overall_nonwest, 2))